# 16. Test local ELECTRA and a neural blend

Owner: Jalil-g. Issue #16. Depends on completed #12 and #15. Blend subsection waits for #17.

## Goal

Fine-tune a small generic ELECTRA locally and see if a neural model adds anything beyond the lexical and style arms. Select the epoch count on internal 6400/1600, then confirm once on frozen 2000.

## Setup

Runs in the optional isolated env `envs/electra` (torch 2.15.0, transformers 5.22.2, huggingface-hub 2.4.0). Main `uv.lock` stays torch-free. Checkpoint is `google/electra-small-discriminator` at revision `fa8239aadc095e9164941d05878b98afe9b953c3`, Apache 2.0. Weights are downloaded once into ignored `artifacts/`, then everything loads with `local_files_only=True`. No remote inference.

In [ ]:
import os, json, time, math, textwrap, traceback, platform, subprocess, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
from sklearn.metrics import roc_auc_score
from IPython import get_ipython
get_ipython().run_line_magic("run", "./02_split_metrics.ipynb")  # helpers only, no fits
hf_logging.set_verbosity_error()

NAME = "16_local_electra"
COMMAND = "uv run --project envs/electra --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=7200 notebooks/16_local_electra.ipynb"
REPO_ID = "google/electra-small-discriminator"
REVISION = "fa8239aadc095e9164941d05878b98afe9b953c3"
CHECKPOINT_FILES = ["config.json", "pytorch_model.bin", "tokenizer.json", "tokenizer_config.json", "vocab.txt"]
MAX_TOKENS = 256
BATCH = 16
EVAL_BATCH = 64
LR = 3e-5
WEIGHT_DECAY = .01
WARMUP_FRACTION = .1
SCHEDULE_EPOCHS = 3  # LR horizon for both runs, even if fewer epochs are selected
INNER_EVAL_PER_CLASS = 800
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
# Fixed before #17 exists. Not run in this notebook unless #17 is complete.
BLEND = {"neural_weights": [0, .1, .25, .5, .75, 1], "scores": "standardized with eligible-unlabeled reference",
         "objective": "mean AUROC over original, remove_blank_lines, collapse_whitespace, wrap80",
         "guardrail": "original AUROC >= classical original - .002", "upload": "never automatic"}
print("Device:", DEVICE, "| torch", torch.__version__)

## Checks

Notebook 02 checks data bytes, duplicate audit, and the frozen split. #12 and #15 must have completed records. The internal split uses notebook 02 `split_ids` inside original 8000 with seed 42 and exact 800/800 eval quotas, same as #12. Stress functions are copied from #12 and tested on synthetic strings. Checkpoint files are pinned by revision and hashed.

In [ ]:
START = time.perf_counter()
parts = load_partitions()
train, validation, unlabeled = parts["train"], parts["validation"], parts["unlabeled"]
test = read_raw("test.csv")
validate_ids(test)
require("label" not in test, "Test must be unlabeled")

def completed_record(folder):
    paths = [p for p in sorted((ROOT / "results" / folder).glob("2*Z.json")) if json.loads(p.read_text()).get("status") == "completed"]
    require(bool(paths), f"Completed {folder} record is required")
    return {"record": str(paths[-1].relative_to(ROOT)), "record_sha256": file_hash(paths[-1])}
dependencies = {"12": completed_record("12_seed_stress"), "15": completed_record("15_normalized_transfer")}

# Internal 6400/1600 inside original 8000 only. Same recipe as #12 split seed 42.
_, components, _ = checked_audit(ROOT / parts["split"]["audit_path"])
train_ids = set(train.id)
inner_components = [[row for row in comp if row["file"] == "train.csv" and row["id"] in train_ids] for comp in components]
inner_components = [comp for comp in inner_components if len(comp) > 1]
fit_ids, eval_ids = split_ids(train[["id", "label"]], inner_components, validation_per_class=INNER_EVAL_PER_CLASS, seed=SEED)
indexed = train.set_index("id", drop=False)
inner_fit, inner_eval = indexed.loc[fit_ids].reset_index(drop=True), indexed.loc[eval_ids].reset_index(drop=True)
require(len(inner_fit) == 6400 and len(inner_eval) == 1600, "Bad inner sizes")
require(inner_eval.label.value_counts().to_dict() == {0: 800, 1: 800}, "Bad inner class quota")
require(not set(inner_eval.id) & set(validation.id), "Inner eval touched frozen validation")
inner_digest = {"fit_ids_sha256": value_hash(fit_ids), "eval_ids_sha256": value_hash(eval_ids)}
record12 = json.loads((ROOT / dependencies["12"]["record"]).read_text())
require(record12["config"]["inner_split_digests"]["42"] == inner_digest, "Inner split differs from #12 split seed 42")

def remove_blank_lines(text):
    return "\n".join(line for line in str(text).split("\n") if line.strip())
def collapse_whitespace(text):
    return " ".join(str(text).split())
def wrap80(text):
    # Wrap each nonblank line at 80 columns; blank lines stay so only line length changes.
    return "\n".join(textwrap.fill(line, 80) if line.strip() else line for line in str(text).split("\n"))
STRESSES = {"original": str, "remove_blank_lines": remove_blank_lines, "collapse_whitespace": collapse_whitespace, "wrap80": wrap80}
sample = "a  b\n\n\nc\t d\n" + "word " * 30
require(remove_blank_lines("a\n\n\nb\n  \nc") == "a\nb\nc", "Blank-line removal failed")
require(collapse_whitespace(sample).startswith("a b c d word"), "Whitespace collapse failed")
require(all(len(line) <= 80 for line in wrap80(sample).split("\n")) and "\n\n" in wrap80(sample), "wrap80 failed")

# Pinned checkpoint, downloaded once into ignored artifacts. Training and scoring load it locally only.
CKPT = ROOT / "artifacts" / NAME / "checkpoint"
info = HfApi().model_info(REPO_ID, revision=REVISION)
require(info.sha == REVISION, "Checkpoint revision mismatch")
LICENSE = info.card_data.get("license") if info.card_data else None
require(LICENSE == "apache-2.0", "Unexpected checkpoint license")
snapshot_download(REPO_ID, revision=REVISION, allow_patterns=CHECKPOINT_FILES, local_dir=CKPT)
checkpoint_sha256 = {name: file_hash(CKPT / name) for name in CHECKPOINT_FILES}
tokenizer = AutoTokenizer.from_pretrained(CKPT, local_files_only=True)
probe = tokenizer(["word " * 1000], truncation=True, max_length=MAX_TOKENS)["input_ids"][0]
require(len(probe) == MAX_TOKENS and probe[0] == tokenizer.cls_token_id and probe[-1] == tokenizer.sep_token_id, "Truncation check failed")

def hardware():
    chip = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip() if platform.system() == "Darwin" else platform.processor()
    return {"device": DEVICE, "chip": chip, "platform": platform.platform(), "torch_threads": torch.get_num_threads()}

CONFIG = {"checkpoint": {"repo_id": REPO_ID, "revision": REVISION, "license": LICENSE, "source": "Hugging Face Hub, generic pretrained discriminator",
                         "files_sha256": checkpoint_sha256, "loading": "local_files_only after one pinned download"},
    "train": {"max_tokens": MAX_TOKENS, "batch": BATCH, "lr": LR, "weight_decay": WEIGHT_DECAY, "no_decay": "bias and LayerNorm",
              "warmup_fraction": WARMUP_FRACTION, "schedule": "linear warmup then linear decay", "schedule_epochs": SCHEDULE_EPOCHS,
              "optimizer": "torch AdamW", "grad_clip": None, "head": "ElectraForSequenceClassification, 2 labels", "score": "logit1 - logit0",
              "shuffle": "torch.randperm with Generator seed 42 per epoch", "eval_batch": EVAL_BATCH},
    "selection": "max internal original AUROC over epochs 1-3, earliest epoch on ties",
    "confirm": "fresh model on original 8000, selected epoch count, LR schedule still sized for 3 epochs; frozen 2000 scored once",
    "inner_split": {"recipe": "notebook 02 split_ids inside original 8000, seed 42, 800/800, duplicate groups whole", **inner_digest},
    "stresses": {"original": "unchanged", "remove_blank_lines": "drop whitespace-only lines",
                 "collapse_whitespace": "all whitespace runs to one space", "wrap80": "textwrap.fill each nonblank line at 80"},
    "stress_scope": "eval rows only; training text stays original", "blend": BLEND, "dependencies": dependencies,
    "hardware": hardware(), "nondeterminism": "MPS kernels are not bitwise deterministic; deterministic algorithms off; not rerun to measure",
    "neural_versions": {name: importlib.metadata.version(name) for name in ["torch", "transformers", "huggingface-hub", "tokenizers", "safetensors"]},
    "neural_lock": {"path": "envs/electra/uv.lock", "sha256": file_hash(ROOT / "envs/electra/uv.lock")},
    # Code cells only, so the takeaway text can be written after the run without breaking the hash.
    "notebook_code_sha256": value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks/16_local_electra.ipynb").read_text())["cells"] if c["cell_type"] == "code"])}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
private = ROOT / "artifacts" / NAME / RUN_ID
private.mkdir(parents=True, exist_ok=False)
display(pd.DataFrame([{"partition": k, "rows": len(v), "AI": int(v.label.sum()) if "label" in v else None}
                      for k, v in {"inner_fit": inner_fit, "inner_eval": inner_eval, "train8000": train, "frozen2000": validation,
                                   "eligible_unlabeled": unlabeled, "test": test}.items()]))
print("License:", LICENSE, "| checkpoint files hashed:", len(checkpoint_sha256), "| device:", CONFIG["hardware"])

## Method

Pre-tokenize once (truncate to 256 tokens, pad per batch). AdamW at 3e-5, weight decay .01 except bias and LayerNorm. Linear warmup over the first 10% of a three-epoch step count, then linear decay. The confirm run keeps that three-epoch horizon even if it stops early, so the selected epoch sees the same LR it saw in selection. Score is the logit margin. Stresses touch eval text only.

In [ ]:
def encode(texts):
    return tokenizer(list(map(str, texts)), truncation=True, max_length=MAX_TOKENS)["input_ids"]

def to_batch(input_ids):
    batch = tokenizer.pad({"input_ids": input_ids}, return_tensors="pt")
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def margins(model, texts):
    model.eval()
    ids = encode(texts)
    out = []
    for i in range(0, len(ids), EVAL_BATCH):
        logits = model(**to_batch(ids[i:i + EVAL_BATCH])).logits.float()
        out.append((logits[:, 1] - logits[:, 0]).cpu().numpy())
    values = np.concatenate(out)
    require(values.shape == (len(texts),) and np.isfinite(values).all(), "Bad margins")
    return values

def stress_scores(model, frame):
    return {stress: margins(model, frame.text.map(change)) for stress, change in STRESSES.items()}

def train_electra(fit, epochs, eval_frame=None):
    """Fine-tune a fresh copy of the pinned checkpoint. Optionally score eval_frame after each epoch."""
    torch.manual_seed(SEED)  # same head init and shuffle order in both runs
    model = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2, local_files_only=True).to(DEVICE)
    ids, labels = encode(fit.text), torch.tensor(fit.label.to_numpy(), dtype=torch.long)
    steps_per_epoch = math.ceil(len(ids) / BATCH)
    total_steps = steps_per_epoch * SCHEDULE_EPOCHS
    warmup_steps = round(WARMUP_FRACTION * total_steps)
    decay = [p for n, p in model.named_parameters() if not (n.endswith("bias") or "LayerNorm" in n)]
    no_decay = [p for n, p in model.named_parameters() if n.endswith("bias") or "LayerNorm" in n]
    optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.}], lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    generator = torch.Generator().manual_seed(SEED)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        order, losses, start = torch.randperm(len(ids), generator=generator).tolist(), [], time.perf_counter()
        for i in range(0, len(order), BATCH):
            idx = order[i:i + BATCH]
            loss = model(**to_batch([ids[j] for j in idx]), labels=labels[idx].to(DEVICE)).loss
            loss.backward()
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            losses.append(loss.item())
        if DEVICE == "mps":
            torch.mps.synchronize()
        row = {"epoch": epoch, "steps_done": epoch * steps_per_epoch, "total_schedule_steps": total_steps, "warmup_steps": warmup_steps,
               "lr_end": scheduler.get_last_lr()[0], "mean_train_loss": float(np.mean(losses)), "train_seconds": time.perf_counter() - start}
        if eval_frame is not None:
            start = time.perf_counter()
            scores = stress_scores(model, eval_frame)
            row.update({f"auroc_{s}": float(roc_auc_score(eval_frame.label, v)) for s, v in scores.items()}, eval_seconds=time.perf_counter() - start)
        history.append(row)
        print({k: round(v, 6) if isinstance(v, float) else v for k, v in row.items()})
    return model, history

def select_epoch(history):
    # Max internal original AUROC; earliest epoch wins exact ties.
    best = max(row["auroc_original"] for row in history)
    return min(row["epoch"] for row in history if row["auroc_original"] == best)
require(select_epoch([{"epoch": 1, "auroc_original": .9}, {"epoch": 2, "auroc_original": .95}, {"epoch": 3, "auroc_original": .95}]) == 2, "Tie rule failed")

## Run

Step 1: internal fit 6400, score 1600 after each epoch, pick the epoch, save the pick before touching frozen 2000. Step 2: fresh fit on 8000 for the picked epoch count, score frozen 2000 once under all four stresses. Also score eligible unlabeled and test with the 8000 model for #17 and #18. Row scores and the model stay in ignored `artifacts/`. A failure writes a safe failed record.

In [ ]:
plan_path = ROOT / "results" / NAME / f"{RUN_ID}-plan.json"
save_new_json(plan_path, {"config": CONFIG, "config_sha256": value_hash(CONFIG), "saved_before_fit": True, "split_sha256": parts["split"]["split_sha256"]})
artifacts = [plan_path]
try:
    run_start = time.perf_counter()
    _, inner_history = train_electra(inner_fit, SCHEDULE_EPOCHS, eval_frame=inner_eval)
    inner_seconds = time.perf_counter() - run_start
    SELECTED_EPOCHS = select_epoch(inner_history)
    selection_path = ROOT / "results" / NAME / f"{RUN_ID}-selection.json"
    save_new_json(selection_path, {"selected_epochs": SELECTED_EPOCHS, "inner_history": inner_history, "rule": CONFIG["selection"],
                                   "saved_before_frozen_validation": True, "inner_split": CONFIG["inner_split"]})
    artifacts.append(selection_path)
    print("Selected epochs:", SELECTED_EPOCHS)

    start = time.perf_counter()
    model, confirm_history = train_electra(train, SELECTED_EPOCHS)
    confirm_train_seconds = time.perf_counter() - start
    start = time.perf_counter()
    frozen = stress_scores(model, validation)
    reference_margin, test_margin = margins(model, unlabeled.text), margins(model, test.text)
    confirm_score_seconds = time.perf_counter() - start
    frozen_auroc = {s: float(roc_auc_score(validation.label, v)) for s, v in frozen.items()}

    model_dir = private / "model_train8000"
    model.save_pretrained(model_dir)
    paths = {"frozen2000_margins.csv": pd.DataFrame({"id": validation.id, **{f"margin_{s}": v for s, v in frozen.items()}}),
             "eligible_unlabeled_margins.csv": pd.DataFrame({"id": unlabeled.id, "margin_original": reference_margin}),
             "test_margins.csv": pd.DataFrame({"id": test.id, "margin_original": test_margin})}
    for name, frame in paths.items():
        frame.to_csv(private / name, index=False)
        artifacts.append(private / name)
    artifacts += sorted(p for p in model_dir.iterdir() if p.is_file())
    run_seconds = time.perf_counter() - run_start
except Exception as exc:
    (private / "failure_traceback.txt").write_text(traceback.format_exc())
    record_run(NAME, COMMAND, CONFIG, SEED, 0, 0, {"validation_auroc": None}, time.perf_counter() - START,
               artifacts=artifacts, status="failed", error_type=type(exc).__name__)
    raise RuntimeError("ELECTRA run failed. See safe record and ignored failure_traceback.txt.") from None

## Results

Internal rows are 1600 eval rows inside original 8000. Confirm rows are frozen 2000, scored once after selection. Neither is Kaggle. Loss is original AUROC minus stressed AUROC.

In [ ]:
inner_table = pd.DataFrame(inner_history)
display(inner_table.round(6))
confirm_table = pd.DataFrame([{"stress": s, "frozen2000_auroc": a, "loss_vs_original": frozen_auroc["original"] - a} for s, a in frozen_auroc.items()])
display(confirm_table.round(6))
display(pd.DataFrame(confirm_history).round(6))
timing = {"inner_3_epochs_with_eval_seconds": inner_seconds, "confirm_train_seconds": confirm_train_seconds,
          "confirm_scoring_seconds": confirm_score_seconds, "run_seconds": run_seconds}
print({k: round(v, 1) for k, v in timing.items()}, "| device:", DEVICE)
print("Mean of four frozen stress AUROCs:", round(float(np.mean(list(frozen_auroc.values()))), 6))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for stress in STRESSES:
    axes[0].plot(inner_table.epoch, inner_table[f"auroc_{stress}"], marker="o", label=stress)
axes[0].axvline(SELECTED_EPOCHS, color="gray", ls="--", lw=1)
axes[0].set(title="Internal 1600 AUROC by epoch", xlabel="epoch", ylabel="AUROC", xticks=[1, 2, 3]); axes[0].legend(fontsize=8)
axes[1].bar(confirm_table.stress, confirm_table.frozen2000_auroc, color="tab:blue")
axes[1].set(title=f"Frozen 2000, {SELECTED_EPOCHS} epoch(s) on 8000", ylabel="AUROC", ylim=(max(0, confirm_table.frozen2000_auroc.min() - .05), 1))
axes[1].tick_params(axis="x", labelrotation=20)
fig.tight_layout(); plot_path = ROOT / "results" / NAME / f"{RUN_ID}-electra.png"; fig.savefig(plot_path, dpi=130); artifacts.append(plot_path); plt.show()

### Neural blend (waits for #17)

Fixed plan: neural weights 0, .1, .25, .5, .75, 1 on standardized scores (reference is eligible unlabeled), objective is mean AUROC over original, remove blank lines, collapse whitespace, wrap80, guardrail original >= classical original - .002. It needs #17's completed classical blend. This cell only checks for it and records the status. It never uploads.

In [ ]:
blend17 = sorted((ROOT / "results/17_final_blend").glob("2*Z.json")) if (ROOT / "results/17_final_blend").exists() else []
blend17 = [p for p in blend17 if json.loads(p.read_text()).get("status") == "completed"]
BLEND_STATUS = "not_run_waiting_for_17" if not blend17 else "not_run_17_complete_follow_up_needed"
print("Blend status:", BLEND_STATUS)

## Takeaway

- Tried: TBD after run.
- Found: TBD after run.
- Next: TBD after run.

## Save

Safe aggregates go to `results/16_local_electra/`. The checkpoint copy, fine-tuned model, and row margins stay under ignored `artifacts/16_local_electra/`; only their hashes are recorded. The 8000-row model is a held-out candidate, not a full refit. Full 10000 refit belongs to #18 after final selection.

In [ ]:
metrics = {"validation_auroc": frozen_auroc["original"], "frozen2000_stress_auroc": frozen_auroc,
    "frozen2000_mean_stress_auroc": float(np.mean(list(frozen_auroc.values()))), "selected_epochs": SELECTED_EPOCHS,
    "inner_history": inner_history, "confirm_history": confirm_history, "timing": timing, "blend_status": BLEND_STATUS,
    "scored_rows": {"frozen2000": len(validation), "eligible_unlabeled": len(unlabeled), "test": len(test)}}
record_path = record_run(NAME, COMMAND, CONFIG, SEED, 8000, 2000, metrics, time.perf_counter() - START, artifacts=artifacts)
print("Saved", record_path.relative_to(ROOT))